# Canopy cover: random vs. grid planting

525 trees/ha, each reaching a 4 m crown diameter.

A 4 m crown covers $\pi \cdot 2^2 = 12.57\ \mathrm{m^2}$. At 525 trees/ha the total crown area is $525 \times 12.57 = 6597\ \mathrm{m^2}$ per 10,000 m² → **66% cover if crowns never overlap** (grid). Random placement is a Poisson process, so expected cover is $1 - e^{-\lambda a} \approx 48\%$. We rasterize an actual simulation to confirm.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Parameters ---
trees_per_ha = 525
crown_diam   = 4.0                # m
crown_r      = crown_diam / 2
side         = 100.0             # 1 ha = 100 x 100 m
n_trees      = trees_per_ha
rng          = np.random.default_rng(42)

# --- Grid planting: regular lattice ---
n_side  = int(round(np.sqrt(n_trees)))
spacing = side / n_side
gx, gy  = np.meshgrid((np.arange(n_side) + 0.5) * spacing,
                      (np.arange(n_side) + 0.5) * spacing)
grid_pts = np.column_stack([gx.ravel(), gy.ravel()])[:n_trees]

# --- Random planting: uniform (Poisson) ---
rand_pts = rng.uniform(0, side, size=(n_trees, 2))

# --- Rasterize to measure true canopy cover (accounts for overlap) ---
res     = 0.2                     # m / pixel
n_pix   = int(side / res)
yy, xx  = np.mgrid[0:n_pix, 0:n_pix] * res + res / 2

def cover_fraction(pts):
    canopy = np.zeros((n_pix, n_pix), dtype=bool)
    r2 = crown_r ** 2
    for px, py in pts:
        canopy |= (xx - px) ** 2 + (yy - py) ** 2 <= r2
    return canopy.mean()

grid_cov = cover_fraction(grid_pts)
rand_cov = cover_fraction(rand_pts)

a   = np.pi * crown_r ** 2
lam = trees_per_ha / 1e4
print(f'grid   cover (sim) = {grid_cov*100:4.1f}%   (no-overlap theory {min(1, lam*a)*100:.1f}%)')
print(f'random cover (sim) = {rand_cov*100:4.1f}%   (1 - e^-x theory   {(1-np.exp(-lam*a))*100:.1f}%)')

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, pts, cov, title in [
    (axes[0], grid_pts, grid_cov, 'Grid planting'),
    (axes[1], rand_pts, rand_cov, 'Random planting'),
]:
    ax.set_facecolor('white')
    for px, py in pts:
        ax.add_patch(plt.Circle((px, py), crown_r, color='#2e7d32', alpha=0.55, lw=0))
    ax.set_xlim(0, side); ax.set_ylim(0, side); ax.set_aspect('equal')
    ax.set_title(f'{title}\n{cov*100:.0f}% canopy cover', fontsize=13)
    ax.set_xlabel('m'); ax.set_ylabel('m')

fig.suptitle(f'{trees_per_ha} trees/ha, {crown_diam:.0f} m crown diameter', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()